### Data Cleaning & Feature Engineering

Approach:
- Feature window: Jan–Dec 2017 (12 months) — used to build predictive features
- Outcome window: Jan–Dec 2018 (12 months) — used to define churn
- This separation avoids data leakage entirely.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

loyalty = pd.read_csv("../data/raw/Customer_Loyalty_History.csv")
flights = pd.read_csv("../data/raw/Customer_Flight_Activity.csv")

print(loyalty.shape, flights.shape)

(16737, 16) (392936, 8)


### 1. Handle missing Salary

25% of customers have missing salary. We impute using the median salary
within the same Education group, and add a flag column so the model can
learn that "unknown salary" might itself be informative.

In [2]:
# Check for negative salaries first
print("Negative salary count:", (loyalty['Salary'] < 0).sum())
print(loyalty['Salary'].describe())

Negative salary count: 20
count     12499.000000
mean      79245.609409
std       35008.297285
min      -58486.000000
25%       59246.500000
50%       73455.000000
75%       88517.500000
max      407228.000000
Name: Salary, dtype: float64


In [15]:
loyalty['Salary'] = loyalty.groupby('Education')['Salary'].transform(
    lambda x: x.fillna(x.median())
)

# Fallback: if any group had ALL missing salaries, fill remaining with overall median
loyalty['Salary'] = loyalty['Salary'].fillna(loyalty['Salary'].median())

print("Remaining missing salary:", loyalty['Salary'].isna().sum())

Remaining missing salary: 0


### 2. Build a Year-Month lookup with Quarter and Season

The Calendar file is daily and doesn't include explicit quarter/season labels,
so we derive them ourselves from the month number.
Seasons (Canada): Winter=Dec/Jan/Feb, Spring=Mar/Apr/May, Summer=Jun/Jul/Aug, Fall=Sep/Oct/Nov.

In [4]:
def get_quarter(month):
    return (month - 1) // 3 + 1

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

flights['Quarter'] = flights['Month'].apply(get_quarter)
flights['Season'] = flights['Month'].apply(get_season)
flights.head()

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Quarter,Season
0,100590,2018,6,12,15276,22914.0,0,0,2,Summer
1,100590,2018,7,12,9168,13752.0,0,0,3,Summer
2,100590,2018,5,4,6504,9756.0,0,0,2,Spring
3,100590,2018,10,0,0,0.0,512,92,4,Fall
4,100590,2018,2,0,0,0.0,0,0,1,Winter


### 3. Split flight activity into Feature window (2017) and Outcome window (2018)

In [5]:
flights_2017 = flights[flights['Year'] == 2017].copy()
flights_2018 = flights[flights['Year'] == 2018].copy()

print("2017 rows:", flights_2017.shape[0])
print("2018 rows:", flights_2018.shape[0])
print("Customers with 2017 data:", flights_2017['Loyalty Number'].nunique())
print("Customers with 2018 data:", flights_2018['Loyalty Number'].nunique())

2017 rows: 191100
2018 rows: 201836
Customers with 2017 data: 15766
Customers with 2018 data: 16737


### 4. Build 2017 features (per customer)

These describe behavior BEFORE the outcome period — safe to use for prediction.

In [6]:
agg_2017 = flights_2017.groupby('Loyalty Number').agg(
    total_flights_2017=('Total Flights', 'sum'),
    total_distance_2017=('Distance', 'sum'),
    total_points_accum_2017=('Points Accumulated', 'sum'),
    total_points_redeemed_2017=('Points Redeemed', 'sum'),
    total_dollar_redeemed_2017=('Dollar Cost Points Redeemed', 'sum'),
    avg_monthly_flights_2017=('Total Flights', 'mean'),
    std_monthly_flights_2017=('Total Flights', 'std'),
    active_months_2017=('Total Flights', lambda x: (x > 0).sum())
).reset_index()

# Redemption rate (avoid division by zero)
agg_2017['redemption_rate_2017'] = np.where(
    agg_2017['total_points_accum_2017'] > 0,
    agg_2017['total_points_redeemed_2017'] / agg_2017['total_points_accum_2017'],
    0
)

# Seasonal flight totals
season_pivot = flights_2017.pivot_table(
    index='Loyalty Number', columns='Season', values='Total Flights', aggfunc='sum', fill_value=0
).reset_index()
season_pivot.columns = ['Loyalty Number'] + [f'flights_{c.lower()}_2017' for c in season_pivot.columns[1:]]

# Recency: last month with flight activity in 2017 (12 = December = most recent)
last_active = flights_2017[flights_2017['Total Flights'] > 0].groupby('Loyalty Number')['Month'].max().reset_index()
last_active.columns = ['Loyalty Number', 'last_active_month_2017']
last_active['months_since_last_flight_2017'] = 12 - last_active['last_active_month_2017']

agg_2017 = agg_2017.merge(season_pivot, on='Loyalty Number', how='left')
agg_2017 = agg_2017.merge(last_active[['Loyalty Number', 'months_since_last_flight_2017']], on='Loyalty Number', how='left')

# If customer had NO flights all of 2017, they never flew -> recency = 12 (max)
agg_2017['months_since_last_flight_2017'] = agg_2017['months_since_last_flight_2017'].fillna(12)
agg_2017['std_monthly_flights_2017'] = agg_2017['std_monthly_flights_2017'].fillna(0)

agg_2017.head()

,Loyalty Number,total_flights_2017,total_distance_2017,total_points_accum_2017,total_points_redeemed_2017,total_dollar_redeemed_2017,avg_monthly_flights_2017,std_monthly_flights_2017,active_months_2017,redemption_rate_2017,flights_fall_2017,flights_spring_2017,flights_summer_2017,flights_winter_2017,months_since_last_flight_2017
0,100018,24,37760,37760.0,1128,203,2.000000,1.414214,10,0.029873,9,5,7,3,0.0
1,100102,25,39884,39884.0,1195,215,2.083333,1.781640,8,0.029962,7,8,4,6,0.0
2,100140,22,31582,31582.0,0,0,1.833333,1.696699,9,0.000000,6,6,3,7,0.0
3,100214,10,18186,18186.0,861,155,0.833333,1.267304,4,0.047344,5,0,0,5,0.0
4,100272,20,28486,28486.0,393,71,1.666667,2.059715,6,0.013796,1,9,7,3,0.0


### 5. Build 2018 outcome data (used ONLY to define churn label, NOT as a feature)

In [7]:
agg_2018 = flights_2018.groupby('Loyalty Number').agg(
    total_flights_2018=('Total Flights', 'sum')
).reset_index()

# Flights in H2 2018 (Jul-Dec) - used for behavioral churn definition
h2_2018 = flights_2018[flights_2018['Month'] >= 7].groupby('Loyalty Number')['Total Flights'].sum().reset_index()
h2_2018.columns = ['Loyalty Number', 'flights_h2_2018']

agg_2018 = agg_2018.merge(h2_2018, on='Loyalty Number', how='left')
agg_2018['flights_h2_2018'] = agg_2018['flights_h2_2018'].fillna(0)

agg_2018.head()

,Loyalty Number,total_flights_2018,flights_h2_2018
0,100018,22,17
1,100102,26,17
2,100140,25,11
3,100214,12,9
4,100272,17,9


### 6. Define churn label

churned = 1 if:
  (a) Cancellation Year == 2018  OR
  (b) zero flights in Jul-Dec 2018 (behavioral disengagement)
else 0

In [8]:
# Merge everything together
df = loyalty.merge(agg_2017, on='Loyalty Number', how='left')
df = df.merge(agg_2018, on='Loyalty Number', how='left')

# Customers with no 2017 flight records at all -> fill activity features with 0
fill_cols_2017 = [c for c in agg_2017.columns if c != 'Loyalty Number']
df[fill_cols_2017] = df[fill_cols_2017].fillna(0)

fill_cols_2018 = [c for c in agg_2018.columns if c != 'Loyalty Number']
df[fill_cols_2018] = df[fill_cols_2018].fillna(0)

# Churn label
cancelled_2018 = (df['Cancellation Year'] == 2018)
no_h2_activity = (df['flights_h2_2018'] == 0)

df['churned'] = (cancelled_2018 | no_h2_activity).astype(int)

print(df['churned'].value_counts())
print("\nChurn rate:", df['churned'].mean().round(3))

churned
0    13943
1     2794
Name: count, dtype: int64

Churn rate: 0.167


### 7. Add tenure feature (membership length as of end of 2017)

This is known information at prediction time — how long someone has
been a loyalty member is a classic churn predictor.

In [9]:
# Tenure in months, measured up to Dec 2017 (our prediction point)
df['tenure_months'] = (2017 - df['Enrollment Year']) * 12 + (12 - df['Enrollment Month'])

# Safety: tenure can't be negative (enrolled after Dec 2017 shouldn't exist, but check)
print("Negative tenure count:", (df['tenure_months'] < 0).sum())
df['tenure_months'] = df['tenure_months'].clip(lower=0)

df[['Loyalty Number', 'Enrollment Year', 'Enrollment Month', 'tenure_months']].head()

Negative tenure count: 3010


,Loyalty Number,Enrollment Year,Enrollment Month,tenure_months
0,480934,2016,2,22
1,549612,2016,3,21
2,429460,2014,7,41
3,608370,2013,2,58
4,530508,2014,10,38


### 8. Drop columns that would cause data leakage or aren't useful

- Cancellation Year/Month: directly encode the 2018 outcome -> LEAKAGE, drop
- total_flights_2018, flights_h2_2018: outcome-window data -> drop after using for label
- City, Postal Code: too granular, high cardinality, not generalizable

In [10]:
leakage_cols = ['Cancellation Year', 'Cancellation Month', 'total_flights_2018', 'flights_h2_2018']
df_model = df.drop(columns=leakage_cols + ['City', 'Postal Code'])

print(df_model.shape)
df_model.columns.tolist()

(16737, 29)


['Loyalty Number',
 'Country',
 'Province',
 'Gender',
 'Education',
 'Salary',
 'Marital Status',
 'Loyalty Card',
 'CLV',
 'Enrollment Type',
 'Enrollment Year',
 'Enrollment Month',
 'Salary_Missing',
 'total_flights_2017',
 'total_distance_2017',
 'total_points_accum_2017',
 'total_points_redeemed_2017',
 'total_dollar_redeemed_2017',
 'avg_monthly_flights_2017',
 'std_monthly_flights_2017',
 'active_months_2017',
 'redemption_rate_2017',
 'flights_fall_2017',
 'flights_spring_2017',
 'flights_summer_2017',
 'flights_winter_2017',
 'months_since_last_flight_2017',
 'churned',
 'tenure_months']

### 9. Save processed dataset

In [11]:
df_model.to_csv("../data/processed/customer_master.csv", index=False)
print("Saved:", df_model.shape)
df_model.head()

Saved: (16737, 29)


,Loyalty Number,Country,Province,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Salary_Missing,total_flights_2017,total_distance_2017,total_points_accum_2017,total_points_redeemed_2017,total_dollar_redeemed_2017,avg_monthly_flights_2017,std_monthly_flights_2017,active_months_2017,redemption_rate_2017,flights_fall_2017,flights_spring_2017,flights_summer_2017,flights_winter_2017,months_since_last_flight_2017,churned,tenure_months
0,480934,Canada,Ontario,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,0,18.0,25352.0,25352.0,322.0,58.0,1.500000,1.678744,7.0,0.012701,2.0,4.0,10.0,2.0,0.0,0,22
1,549612,Canada,Alberta,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,1,31.0,36136.0,36136.0,787.0,142.0,2.583333,2.020726,8.0,0.021779,7.0,9.0,9.0,6.0,1.0,0,21
2,429460,Canada,British Columbia,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,1,16.0,22957.0,22957.0,374.0,67.0,1.333333,1.370689,7.0,0.016291,4.0,6.0,3.0,3.0,1.0,1,41
3,608370,Canada,Ontario,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,1,24.0,33369.0,33369.0,864.0,156.0,2.000000,1.758098,9.0,0.025892,5.0,10.0,5.0,4.0,0.0,0,58
4,530508,Canada,Quebec,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,0,11.0,19886.0,19886.0,0.0,0.0,0.916667,1.443376,4.0,0.000000,2.0,2.0,4.0,3.0,1.0,0,38


### 10. Separate 2018 enrollees (no 2017 history) from the churn modeling set

Customers who enrolled in 2018 have zero 2017 activity by definition (not
because they're inactive). Including them would mislead the model. We
split them into a separate "new members" group for a different business
treatment (onboarding focus, not churn risk).

In [12]:
new_members_2018 = df_model[df_model['Enrollment Year'] == 2018].copy()
churn_model_data = df_model[df_model['Enrollment Year'] != 2018].copy()

print("New members (enrolled 2018):", new_members_2018.shape[0])
print("Existing members for churn modeling:", churn_model_data.shape[0])
print("\nChurn rate in modeling set:", churn_model_data['churned'].mean().round(3))

New members (enrolled 2018): 3010
Existing members for churn modeling: 13727

Churn rate in modeling set: 0.163


In [13]:
churn_model_data.to_csv("../data/processed/churn_model_data.csv", index=False)
new_members_2018.to_csv("../data/processed/new_members_2018.csv", index=False)
print("Saved both files.")

Saved both files.


In [14]:
print(df_model['Country'].unique())
print(df_model['Province'].nunique(), df_model['Province'].unique())

<ArrowStringArray>
['Canada']
Length: 1, dtype: str
11 <ArrowStringArray>
[             'Ontario',              'Alberta',     'British Columbia',
               'Quebec',                'Yukon',        'New Brunswick',
             'Manitoba',          'Nova Scotia',         'Saskatchewan',
         'Newfoundland', 'Prince Edward Island']
Length: 11, dtype: str
